# Look at the flamegraph first

This SVG is generated from a **real folded capture**: `spur graph build --root ../ade-bench` (cold, embeddings off).

| | |
|---|---|
| Wall / user / sys | 6.75s / 4.72s / 1.36s (`/usr/bin/time -l`) |
| Mode | `Full` — 1,461 files, 12,237 nodes (python + markdown) |
| Sampler | `samply record --rate 4000` (macOS, no sudo) |
| Folded | `test/data/spur-ade-bench.folded` — 25,782 on-CPU samples, 2,289 unique stacks |
| SVG | `test/data/spur-ade-bench.svg` — written from that folded file |

Width is inclusive time; root at the bottom. Hover a frame for name + %.

A flamegraph is the right **picture**. It is the wrong **query surface** for a data-analyst performance investigation — that is why `quack_flamegraph` loads the same stacks into DuckDB.

In [9]:
"""Rebuild the SVG from the live folded file and display it."""
import subprocess
import sys
from pathlib import Path

from IPython.display import HTML, display


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "Cargo.toml").exists() and (p / "test" / "data" / "spur-ade-bench.folded").exists():
            return p
    raise FileNotFoundError(f"repo root not found from cwd={here}")


ROOT = repo_root()
FOLDED = ROOT / "test/data/spur-ade-bench.folded"
SVG = ROOT / "test/data/spur-ade-bench.svg"
SCRIPT = ROOT / "scripts/folded_to_svg.py"

r = subprocess.run(
    [sys.executable, str(SCRIPT), str(FOLDED), "-o", str(SVG),
     "--title", "spur graph build --root ../ade-bench (cold, no embeddings)"],
    check=True, capture_output=True, text=True,
)
print(r.stdout.strip())
display(HTML(
    "<div style='font-family:ui-sans-serif,system-ui,sans-serif;margin:0 0 8px 0'>"
    "<div style='font-size:13px;color:#444;margin:0 0 6px 0'>"
    "Live folded → SVG: <code>test/data/spur-ade-bench.folded</code> → "
    "<code>test/data/spur-ade-bench.svg</code>"
    "</div>"
    f"{SVG.read_text()}"
    "</div>"
))


samples 25782 stacks 2289 wrote /Volumes/Projects/Projects/quack-flamegraph/test/data/spur-ade-bench.svg


## Why a data analyst still needs `quack_flamegraph`

Hover the fat plateau through `extract_files` / `build_facts` (~74% coverage) and the hottest **leaf** `ts_parser_parse` (~9% exclusive). That is a useful first look. It is not a decision procedure.

On this live ade-bench capture the picture already misleads in the usual ways:

- The C parser leaf looks like “the bug”. Inclusive coverage says most stacks are inside `spur_graph::extract::tree_sitter::extract_files` (~74%), and Python re-export resolution is ~19% — a product path, not `ts_lex`.
- Widths overlap. You cannot ask “what is left if we skip re-export resolution?” without a `WHERE`.
- Exclusive vs coverage vs parent→child edges are different rankings. The SVG shows one of them (inclusive width).

| Question you actually have to answer | What the SVG gives you | What `quack_flamegraph` gives you |
|---|---|---|
| What % of *unique stacks* contain this frame? | A guessed width (inclusive, overlapping) | `flamegraph_coverage` — each stack counted once |
| Did a recursive name inflate the hotspot? | Easy to miss stacked same-name frames | Join coverage vs `flamegraph_inclusive` |
| What is the hottest *parent → child* edge? | Adjacency you squint at | `flamegraph_edges` |
| Hottest leaf vs product path? | Leaf name (`ts_parser_parse`, a C helper) | Coverage of `extract_files` / Python re-export |
| If we drop that path, what is left? | Record a new profile | `WHERE NOT list_contains(frames, …)` then re-chart |
| Do buckets sum to 100%? | No — widths overlap | Disjoint `CASE` over unique stacks |

Squinting at rectangles is how you **start**. Ranking a fix from a join is how you **stop guessing**.

The SQL + GGSQL cells below query **this** capture (`test/data/spur-ade-bench.folded` → `build/perf-demo/*.parquet`).

Loop (do not skip a step):

1. **Time** — 6.75s wall for this capture.
2. **Profile as tables** — `read_folded` → coverage / exclusive / edges.
3. **Quantify** — pick the metric that does not lie.
4. **Fix one thing** — the smallest change those numbers justify.

Community install (DuckDB **v1.5.5**): `INSTALL quack_flamegraph FROM community;` then `LOAD quack_flamegraph;`

In [10]:
import subprocess
import sys
from pathlib import Path


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "Cargo.toml").exists() and (p / "scripts" / "extract_perf_demo.py").exists():
            return p
    raise FileNotFoundError(f"repo root not found from cwd={here}")


ROOT = repo_root()
SCRIPT = ROOT / "scripts/extract_perf_demo.py"
OUT = ROOT / "build/perf-demo"
py = ROOT / "configure/venv/bin/python"
exe = str(py if py.exists() else Path(sys.executable))
# Jute SQL kernel is DuckDB 1.5.4; community quack_flamegraph is 1.5.5.
r = subprocess.run([exe, str(SCRIPT)], check=True, capture_output=True, text=True, cwd=ROOT)
print(r.stdout)
if r.stderr:
    print(r.stderr)
print("parquet", sorted(p.name for p in OUT.glob("*.parquet")))


duckdb 1.5.5 samples 25782 folded spur-ade-bench.folded
parquet ['buckets.parquet', 'coverage.parquet', 'edges.parquet', 'exclusive.parquet', 'fix.parquet', 'inflate.parquet', 'overview.parquet', 'pareto.parquet']

parquet ['buckets.parquet', 'coverage.parquet', 'edges.parquet', 'exclusive.parquet', 'fix.parquet', 'inflate.parquet', 'overview.parquet', 'pareto.parquet']


## 1. Same stacks, as a table

`read_folded` turns `test/data/spur-ade-bench.folded` into `frames VARCHAR[]`, `samples BIGINT`. This row is the **inventory**, not a ranking: 25,782 on-CPU samples, 2,289 unique stacks, mean depth 19.1 (max 50).

GGSQL is off here on purpose — a 1-row summary is a table. Charts start when there is a distribution to read.

In [16]:
SELECT samples, stacks, avg_depth, max_depth
FROM read_parquet('../../../build/perf-demo/overview.parquet');

samples,stacks,avg_depth,max_depth
25782,2289,19.1,50


## 2. Unique-stack coverage — do not add the bars

Flamegraph **width is inclusive**. `flamegraph_coverage` counts each unique stack **once**, so a frame's bar is “% of samples whose stack contains this frame.”

**How to read the bar chart:** taller = more stacks pass through that frame. Bars **overlap** (a parse stack is also an `extract_files` stack), so the percentages must not be summed. Near-100% runtime frames (`main`, `lang_start`) are filtered out so the product envelope is visible.

On this live capture the envelope is `build_facts` / `extract_files` (~74%). Tree-sitter parse is 31%. Python re-export resolution (`reexport_source_candidates` → `module_file_matches_suffix`) is ~19% — a Spur path, not C.

In [15]:
SELECT frame, coverage, n_stacks, pct
FROM read_parquet('../../../build/perf-demo/coverage.parquet')
ORDER BY pct DESC;

frame,coverage,n_stacks,pct
extract::tree_sitter::build_facts,19210,944,74.51
extract::tree_sitter::extract_files,19105,922,74.10
ts_parser_parse_with_options,8047,344,31.21
ts_parser_parse,8045,343,31.20
...ree_sitter::FactBuilder>::resolve_pending_edges,4965,157,19.26
extract::tree_sitter::resolve_bare_pending_edge,4868,107,18.88
extract::tree_sitter::reexport_source_candidates,4818,81,18.69
...sitter::reexport_resolution_candidates_at_depth,4818,81,18.69
...rt_source_candidates::{closure#2}>>>::from_iter,4814,77,18.67
extract::tree_sitter::module_file_matches_suffix,4747,77,18.41


**Inflate chart (next).** `flamegraph_inclusive` unnests, so a recursive/closure name on the same stack is counted once **per occurrence**. `inflate = inclusive / coverage`. On this capture the only material inflate (≥1.05 at ≥1% coverage) is Lance’s inverted-index builder at **2.0** (7.1% coverage looks like 14% inclusive). Rank by coverage, not inclusive.

In [13]:
SELECT frame, coverage, inclusive, inflate, coverage_pct, inclusive_pct
FROM read_parquet('../../../build/perf-demo/inflate.parquet')
ORDER BY inflate DESC, coverage DESC;

frame,coverage,inclusive,inflate,coverage_pct,inclusive_pct
...io::runtime::scheduler::current_thread::Handle>>>::poll,1830,3660,2.0,7.10,14.20
...::runtime::blocking::schedule::BlockingSchedule>>::poll,444,888,2.0,1.72,3.44


**Pareto of unique stacks.** X is stack rank (fattest first); Y is cumulative % of all samples. A steep early rise means a few fat stacks dominate. Use it to decide whether a fix on one spine (re-export, parse) moves the wall-clock number, or whether time is smeared across thousands of tiny stacks.

In [14]:
SELECT rank, cum_pct, samples, leaf
FROM read_parquet('../../../build/perf-demo/pareto.parquet')
ORDER BY rank;

rank,cum_pct,samples,leaf
1,7.77,2004,ts_parser_parse
2,13.65,1516,ts_parser__do_all_potential_reductions
3,17.26,931,ts_query__perform_analysis
4,20.20,758,store::build::buckets_from_facts
5,22.38,560,__open
6,24.40,521,ts_query__perform_analysis
7,26.39,515,__posix_spawn
8,28.21,469,...snowball::algorithms::italian::Context>
9,29.76,398,ts_query_new
10,31.15,358,_platform_memmove


## 3. Exclusive leaves lie — walk the edge

Hottest **leaf** is `ts_parser_parse` at 9.2% self-time. That is tree-sitter C, not a product function. Exclusive self-time under LTO/ICF also names helpers (`ts_parser__do_all_potential_reductions`, `ts_query__perform_analysis`).

**Exclusive bar chart:** height = % of samples where this frame is the **leaf**. Use it to find on-CPU sit points, then distrust surprising C names until you see the parent.

**Edge heatmap:** fill = % of all samples on that parent → child step. The two hot columns under `extract_files` split the work: parse (~24%) vs `resolve_pending_edges` → Python re-export (~19%). That split is the decision, not the 9% parse leaf.

In [12]:
SELECT leaf, exclusive, pct
FROM read_parquet('../../../build/perf-demo/exclusive.parquet')
ORDER BY pct DESC;

leaf,exclusive,pct
ts_parser_parse,2379,9.23
ts_parser__do_all_potential_reductions,1532,5.94
ts_query__perform_analysis,1452,5.63
__open,929,3.60
_nanov2_free,904,3.51
_platform_memmove,848,3.29
store::build::buckets_from_facts,758,2.94
nanov2_malloc,747,2.90
ts_subtree_summarize_children,724,2.81
ts_tree_cursor_child_iterator_next,708,2.75


In [17]:
SELECT parent, child, samples, pct
FROM read_parquet('../../../build/perf-demo/edges.parquet')
ORDER BY pct DESC;

parent,child,samples,pct
ts_parser_parse_with_options,ts_parser_parse,8045,31.20
extract::tree_sitter::extract_files,ts_parser_parse_with_options,6133,23.79
extract::tree_sitter::extract_files,...FactBuilder>::resolve_pending_edges,4965,19.26
...FactBuilder>::resolve_pending_edges,...e_sitter::resolve_bare_pending_edge,4868,18.88
...e_sitter::resolve_bare_pending_edge,...port_resolution_candidates_at_depth,4818,18.69
...port_resolution_candidates_at_depth,..._sitter::reexport_source_candidates,4818,18.69
..._sitter::reexport_source_candidates,...ndidates::{closure#2}>>>::from_iter,4814,18.67
...ndidates::{closure#2}>>>::from_iter,..._sitter::module_file_matches_suffix,4746,18.41
..._sitter::module_file_matches_suffix,alloc::fmt::format::format_inner,3693,14.32
...ce<()>>::call_once::{shim:vtable#0},"...ctSidecarRowCounts, anyhow::Error>>",3315,12.86


In [11]:
SELECT bucket, samples, n_stacks, pct
FROM read_parquet('../../../build/perf-demo/buckets.parquet')
ORDER BY pct DESC;

bucket,samples,n_stacks,pct
C_extract_parse,8047,344,31.21
E_extract_other,6342,517,24.60
A_python_reexport,4819,82,18.69
G_other,4018,1127,15.58
D_buckets,1663,84,6.45
F_spawn_io,893,135,3.46


## 4. One fix the numbers justify

**Bucket bars sum to 100%.** Each unique stack is assigned exactly one label (priority: Python re-export → parse → buckets → other extract → spawn/IO → other). This is the chart to compare magnitudes without double-counting.

| Rank | What the SVG/leaf says | What the SQL says | Action |
|---|---|---|---|
| 1 | Hottest leaf `ts_parser_parse` (9%) | Disjoint parse bucket 31%, but it sits *inside* extract; re-export is a separate **18.7%** bucket | **Cap/cache Python `module_file_matches_suffix` / re-export candidate walk** |
| 2 | `buckets_from_facts` exclusive 2.9% | Disjoint bucket D 6.5% | Re-profile after #1 |
| — | Parse looks huge on the flamegraph | It is the C sit-point of extract, not an independent product bug | Do not start here |

The modeled fix (next chart) drops stacks in the re-export walk: **18.7% removed, 81.3% remaining**. That is a SQL counterfactual, not a patch.

In [18]:
SELECT slice, samples, pct
FROM read_parquet('../../../build/perf-demo/fix.parquet')
ORDER BY samples DESC;

slice,samples,pct
remaining,20963,81.31
removed_if_reexport_capped,4819,18.69


## Decision: attack one thing, measure two

Wall-clock is **6.75s**, almost all on-CPU (4.72 user + 1.36 sys). This is a compute problem, not an off-CPU wait problem.

**Attack now (one change)**

1. **Python re-export candidate walk — 18.7% disjoint.** Edge chain `extract_files` → `resolve_pending_edges` → `reexport_source_candidates` → `module_file_matches_suffix` (~14% of that is `format_inner`). This is Spur code. Cap, cache, or index module-suffix matches so import resolution is not a linear scan + format per candidate. SQL already models **−18.7% samples** if that spine goes away.

**Do not attack yet (measure more)**

2. **Parse-inside-extract — 31% of stacks, 9.2% exclusive `ts_parser_parse`.** This is tree-sitter C sitting *inside* extract, not a separate product function. Before touching the grammar, split:
   - `ts_query_new` (2.2% exclusive) — are queries compiled **per file** instead of once per language?
   - `ts_query__perform_analysis` (5.6%) vs `ts_lex` (2.3%) vs reductions (5.9%) — compile vs scan vs parse.
   A query-cache is a one-line experiment; rewriting parse is not.

3. **Other extract (24.6% bucket)** — `build_facts` leftover after parse + re-export. Need a coverage slice of *this* bucket alone (pending-edge work that is not re-export, markdown, SQL extract). Do not optimize “extract” as a blob.

**Later / after the first re-profile**

4. **`buckets_from_facts` — 6.5% disjoint, 2.9% exclusive.** Real, but 3× smaller than re-export. Re-measure after #1; it may grow or vanish in rank.
5. **`__open` / `posix_spawn` — 2.5–3.6%.** Not the wall. Ignore until on-CPU extract is cut.
6. **Lance inflate 2.0 (7.1% coverage).** Inclusive looks like 14%. Ranking trap, not a fix. Keep using coverage.

**How to read the next chart:** color is the decision (`ATTACK now` / `MEASURE more` / `Later`). Height is the disjoint bucket % (bars *do* sum to ~100%). Attack the orange 18.7% first; the 31% parse bar is the thing to *instrument*, not rewrite.

In [19]:
SELECT CASE bucket
         WHEN 'A_python_reexport' THEN 'ATTACK now'
         WHEN 'C_extract_parse' THEN 'MEASURE more'
         WHEN 'E_extract_other' THEN 'MEASURE more'
         WHEN 'D_buckets' THEN 'Later'
         WHEN 'F_spawn_io' THEN 'Later'
         ELSE 'Ignore v1'
       END AS action,
       bucket AS item,
       samples,
       pct
FROM read_parquet('../../../build/perf-demo/buckets.parquet')
ORDER BY pct DESC;

action,item,samples,pct
MEASURE more,C_extract_parse,8047,31.21
MEASURE more,E_extract_other,6342,24.60
ATTACK now,A_python_reexport,4819,18.69
Ignore v1,G_other,4018,15.58
Later,D_buckets,1663,6.45
Later,F_spawn_io,893,3.46


## 5. Re-profile

After the re-export cap lands, capture a new folded file (`samply record --rate 4000`) and re-run this notebook. The next bottleneck should be parse-inside-extract (~31% bucket) or `buckets_from_facts` (~6.5%) — **measured**, not guessed.

```sql
INSTALL quack_flamegraph FROM community;
LOAD quack_flamegraph;
SELECT * FROM flamegraph_coverage('test/data/spur-ade-bench.folded')
ORDER BY coverage DESC LIMIT 12;
```